# Frozen standalone CNN evaluation
Run after all three checkpoints are selected. Fixed threshold and prespecified metrics/diagnostics; no training or tuning. Exactly six essential PNG plots and one final report.

In [1]:
from pathlib import Path
import json, ast
ROOT=(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'dataset_2.csv').is_file()) / 'cnn/cnn wave/eew_causal_v3');OUT=ROOT/'cnn_eew_training'
# Reuse only the common definitions cell, never execute the training cell.
nb=json.loads((OUT/'01_cnn_training.ipynb').read_text(encoding='utf-8'))
exec(compile(''.join(nb['cells'][1]['source']),'training_common_definitions_only','exec'))
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,confusion_matrix,mean_absolute_error,mean_squared_error,r2_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
assert all((OUT/'models'/f'cnn_{h}s_best.pt').exists() for h in [1,2,3])
assert not (OUT/'final_cnn_eew_report.md').exists(),'Existing evaluation report; do not overwrite'
checkpoints={h:torch.load(OUT/'models'/f'cnn_{h}s_best.pt',map_location='cpu',weights_only=False) for h in [1,2,3]}
for ck in checkpoints.values():assert all(sha(Path(p))==v for p,v in ck['input_hashes'].items())
@torch.inference_mode()
def infer(model,x):
 logits=[];magnitudes=[]
 for chunk in x.split(256):
  a,b=model(chunk.to(DEVICE));logits.append(a.cpu());magnitudes.append(b.cpu())
 return torch.cat(logits).sigmoid().numpy(),torch.cat(magnitudes).numpy()
def detection(y,p):
 pred=p>=.5
 return dict(accuracy=float(accuracy_score(y,pred)),precision=float(precision_score(y,pred,zero_division=0)),recall=float(recall_score(y,pred,zero_division=0)),f1=float(f1_score(y,pred,zero_division=0)),roc_auc=float(roc_auc_score(y,p)),pr_auc_average_precision=float(average_precision_score(y,p)),confusion_matrix=confusion_matrix(y,pred,labels=[0,1]).tolist())
results={};plotdata={}


Device: cuda ; parameters: 14674


In [2]:
for h,ck in checkpoints.items():
 model=EarlyCNN().to(DEVICE);model.load_state_dict(ck['state_dict'],strict=True);model.eval()
 # Restore-selected validation loss is checked before accessing test values.
 val=load_split('validation',h);restored=epoch(model,loader(val,False),ck['target_center'],ck['target_scale'])
 assert abs(restored['total']-ck['best_validation_loss'])<1e-6
 test=load_split('test',h);prob,z=infer(model,test['x']);pred=z*ck['target_scale']+ck['target_center']
 y=test['y'].numpy();reg=test['reg'].numpy();actual=test['mag'].numpy()[reg];estimate=pred[reg];residual=estimate-actual
 det=detection(y,prob)
 mag={'mae':float(mean_absolute_error(actual,estimate)),'rmse':float(mean_squared_error(actual,estimate)**.5),'r2':float(r2_score(actual,estimate)),'bias_pred_minus_actual':float(residual.mean()),'actual_std':float(actual.std()),'predicted_std':float(estimate.std()),'std_ratio':float(estimate.std()/actual.std()),'prediction_slope':float(np.polyfit(actual,estimate,1)[0]),'actual_range':[float(actual.min()),float(actual.max())],'predicted_range':[float(estimate.min()),float(estimate.max())]}
 bins=[]
 for lo,hi in [(-float('inf'),3.),(3.,4.),(4.,5.),(5.,6.),(6.,float('inf'))]:
  mask=(actual>=lo)&(actual<hi);cnt=int(mask.sum());bins.append({'range':f'[{lo},{hi})','n':cnt,'mae':float(np.abs(residual[mask]).mean()) if cnt else None,'bias':float(residual[mask].mean()) if cnt else None})
 strong=actual>=5.
 strong_metrics={'threshold':5.,'n':int(strong.sum()),'underprediction_fraction':float((residual[strong]<0).mean()) if strong.any() else None,'bias':float(residual[strong].mean()) if strong.any() else None,'mae':float(np.abs(residual[strong]).mean()) if strong.any() else None}
 # Baseline uses only training magnitudes and no holdout-derived parameter.
 train=load_split('train',h);trmag=train['mag'][train['reg']].numpy();constant=np.full_like(actual,np.median(trmag))
 baseline={'training_median':float(np.median(trmag)),'test_mae':float(mean_absolute_error(actual,constant))}
 trainprob,trainz=infer(model,train['x']);valprob,valz=infer(model,val['x'])
 gaps={'train_f1':detection(train['y'].numpy(),trainprob)['f1'],'validation_f1':detection(val['y'].numpy(),valprob)['f1'],'train_mag_mae':float(mean_absolute_error(trmag,(trainz*ck['target_scale']+ck['target_center'])[train['reg'].numpy()])),'validation_mag_mae':float(mean_absolute_error(val['mag'][val['reg']].numpy(),(valz*ck['target_scale']+ck['target_center'])[val['reg'].numpy()]))}
 # Prespecified shortcut diagnostic, not a model-selection criterion: retain only pre-P context.
 blank=test['x'].clone();blank[:,:,100:]=0.;pblank,_=infer(model,blank)
 ablation=detection(y,pblank)
 with h5py.File(ROOT/'dataset'/'eew_dataset_test.h5','r') as f:
  rows=test['rows'];s=f['s_arrival_seconds'][:][rows][reg];p=f['p_arrival_seconds'][:][rows][reg]
  intervals=s-p-h;profiles=f['noise_profile'].asstr()[:][rows]
  first_s=f['s_arrival_sample'][:][rows][reg];refs=f['reference_sample'][:][rows][reg]
  assert np.all(refs+100*h<first_s)
  assert np.all(intervals>0),'Nonpositive physical pre-S interval; stop'
 timing={'horizon_seconds':h,'same_station_s_minus_cutoff_seconds':{'n':len(intervals),'min':float(intervals.min()),'p05':float(np.quantile(intervals,.05)),'median':float(np.median(intervals)),'max':float(intervals.max())}}
 # Batch-1, preloaded-device, model-only timing includes fixed pointwise transform, excludes HDF5 IO/transfer.
 sample=test['x'][:1].to(DEVICE)
 with torch.inference_mode():
  for _ in range(25):model(sample)
  if DEVICE.type=='cuda':torch.cuda.synchronize()
  lat=[]
  for _ in range(200):
   start=time.perf_counter();model(sample)
   if DEVICE.type=='cuda':torch.cuda.synchronize()
   lat.append((time.perf_counter()-start)*1000)
 timing['inference_ms']={'median':float(np.median(lat)),'p95':float(np.quantile(lat,.95)),'mean':float(np.mean(lat)),'repeats':200,'batch_size':1,'device':str(DEVICE),'includes':'forward and fixed transform; synchronous completion','excludes':'disk read, host-to-device transfer, trigger and acquisition'}
 noise_errors={}
 for profile in sorted(set(profiles[y==0])):
  mask=(y==0)&(profiles==profile);noise_errors[profile]={'n':int(mask.sum()),'false_positive_rate':float((prob[mask]>=.5).mean())}
 results[h]={'detection':det,'magnitude':mag,'magnitude_bins':bins,'strong_magnitude':strong_metrics,'timing':timing,'training_median_baseline':baseline,'generalization':gaps,'post_P_zeroed_diagnostic':ablation,'noise_profile_errors':noise_errors,'test_eligible':len(y),'test_earthquakes':int(reg.sum()),'test_excluded':test['excluded'],'best_epoch':ck['best_epoch'],'epochs_run':len(ck['history'])}
 plotdata[h]={'actual':actual,'estimate':estimate,'residual':residual,'cm':np.array(det['confusion_matrix'])}
 print(f'{h}s: n={len(y)}, EQ={reg.sum()}, F1={det["f1"]:.4f}, MAE={mag["mae"]:.4f}, RMSE={mag["rmse"]:.4f}, R2={mag["r2"]:.4f}',flush=True)


1s: n=2428, EQ=1214, F1=0.6959, MAE=0.6737, RMSE=0.8322, R2=0.1458
2s: n=2421, EQ=1207, F1=0.7140, MAE=0.6402, RMSE=0.7891, R2=0.2308
3s: n=2414, EQ=1200, F1=0.7576, MAE=0.6130, RMSE=0.7577, R2=0.2898


In [3]:
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
def save(fig,name):
 path=OUT/'plots'/name
 assert not path.exists(),'Do not overwrite an existing plot'
 fig.tight_layout();fig.savefig(path,dpi=300,bbox_inches='tight');plt.close(fig)
fig,axs=plt.subplots(1,3,figsize=(14,4))
for ax,(h,ck) in zip(axs,checkpoints.items()):
 hist=ck['history'];epochs=[r['epoch'] for r in hist]
 ax.plot(epochs,[r['train']['total'] for r in hist],label='Train (epoch-end)');ax.plot(epochs,[r['validation']['total'] for r in hist],label='Validation');ax.axvline(ck['best_epoch'],ls=':',color='black',label='Selected');ax.set(title=f'{h}s post-P',xlabel='Epoch',ylabel='BCE + masked standardized Huber');ax.legend()
save(fig,'training_validation_loss.png')
fig,axs=plt.subplots(1,3,figsize=(14,4))
for ax,(h,d) in zip(axs,plotdata.items()):
 ax.scatter(d['actual'],d['estimate'],s=9,alpha=.35);lo=min(d['actual'].min(),d['estimate'].min());hi=max(d['actual'].max(),d['estimate'].max());ax.plot([lo,hi],[lo,hi],'k--');ax.set(title=f'{h}s: MAE {results[h]["magnitude"]["mae"]:.3f}',xlabel='Actual magnitude',ylabel='Predicted magnitude')
save(fig,'actual_predicted_magnitude.png')
fig,axs=plt.subplots(1,3,figsize=(12,4))
for ax,(h,d) in zip(axs,plotdata.items()):
 ax.imshow(d['cm'],cmap='Blues');ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Noise','Earthquake'],yticklabels=['Noise','Earthquake'],xlabel='Predicted',ylabel='Actual',title=f'{h}s, threshold 0.5')
 for i in range(2):
  for j in range(2):ax.text(j,i,str(d['cm'][i,j]),ha='center',va='center',color='white' if d['cm'][i,j]>d['cm'].max()/2 else 'black')
save(fig,'confusion_matrices.png')
fig,axs=plt.subplots(1,3,figsize=(14,4))
for ax,(h,d) in zip(axs,plotdata.items()):
 ax.hist(d['residual'],bins=35,alpha=.8);ax.axvline(0,color='black',ls='--');ax.set(title=f'{h}s residuals',xlabel='Predicted minus actual magnitude',ylabel='Earthquake count')
save(fig,'magnitude_residuals.png')
for metric,name,title in [('mae','magnitude_mae_comparison.png','Magnitude MAE'),('f1','detection_f1_comparison.png','Detection F1')]:
 vals=[results[h]['magnitude' if metric=='mae' else 'detection'][metric] for h in [1,2,3]];fig,ax=plt.subplots(figsize=(6,4));bars=ax.bar(['1s','2s','3s'],vals,color=['#4477aa','#228833','#cc6677']);ax.bar_label(bars,fmt='%.4f',padding=3);ax.set(xlabel='Post-P observation horizon',ylabel=title,title='Eligible synthetic test records');ax.set_ylim(0,(1.08 if metric=='f1' else max(vals)*1.2));save(fig,name)
# Verify only the known original dependencies; no repository scan.
protection={p:sha(Path(p))==v for p,v in checkpoints[1]['input_hashes'].items()}
assert all(protection.values()),'Original dependency changed'
lines=['# Standalone synthetic CNN EEW results','', '**Date:** 2026-11-13. **Completed:** three selected models and frozen-model test evaluation.','', 'Scope: synthetic oracle-P-aligned same-event classification and magnitude regression. No operational EEW claim, BiLSTM, fusion, Attention, Optuna, dataset regeneration or cleanup.','', '## Fixed protocol','',f"Architecture: {checkpoints[1]['architecture']}; {checkpoints[1]['parameter_count']:,} trainable parameters. Input is only the saved normalized waveform, transformed pointwise by sign(x) log(1+|x|). No magnitude, IDs, timing or eligibility metadata enters forward().",'',f"Settings: `{json.dumps(CFG)}`. Target center and population SD fit separately on eligible training earthquakes. Huber is in these standardized magnitude units. Validation selection minimizes full-set mean BCE plus eligible-earthquake mean Huber. No test metric or threshold tuning. Seed 42 and identical initialization/settings across horizons; one seed, not a multi-seed uncertainty study. Train curves use epoch-end frozen weights, matching validation evaluation mode.",'', 'HDF5 classification masks exclude ineligible records before batching. Magnitude loss is evaluated only on the horizon magnitude mask; sentinel noise targets are excluded. An isolated poison-sentinel and empty-mask loss check passed. Saved masks define the benchmark subset; they are privileged evaluation metadata, not a runtime trigger. Existing train/validation/test assignments remain unchanged.','', '## Selected checkpoints','', '| Horizon | Best epoch / run | Validation loss | Train eligible / EQ | Val eligible / EQ | Fit seconds | Checkpoint |','|---|---:|---:|---|---|---:|---|']
for h,ck in checkpoints.items():
 c=ck['counts'];lines.append(f"| {h}s | {ck['best_epoch']} / {len(ck['history'])} | {ck['best_validation_loss']:.6f} | {c['train']['eligible']} / {c['train']['earthquakes']} | {c['validation']['eligible']} / {c['validation']['earthquakes']} | {ck['training_seconds']:.2f} | models/cnn_{h}s_best.pt |")
lines+=['','## Eligible test detection','', 'PR-AUC is average precision (stepwise PR integration). Fixed probability threshold 0.5. Confusion matrix order is actual rows/predicted columns [noise, earthquake].','', '| Horizon | Eligible / EQ / excluded | Accuracy | Precision | Recall | F1 | ROC-AUC | PR-AUC (AP) | Confusion matrix |','|---|---|---:|---:|---:|---:|---:|---:|---|']
for h,r in results.items():
 d=r['detection'];lines.append(f"| {h}s | {r['test_eligible']} / {r['test_earthquakes']} / {r['test_excluded']} | {d['accuracy']:.5f} | {d['precision']:.5f} | {d['recall']:.5f} | {d['f1']:.5f} | {d['roc_auc']:.5f} | {d['pr_auc_average_precision']:.5f} | {d['confusion_matrix']} |")
lines+=['','## Eligible-earthquake test magnitude','', 'Regression metrics include every eligible earthquake, regardless of detection success; these are not end-to-end alarm-conditioned metrics. Horizon populations differ slightly because S eligibility changes.','', '| Horizon | MAE | RMSE | R² | Bias (pred−actual) | Pred/actual SD | Slope | Training-median baseline MAE |','|---|---:|---:|---:|---:|---:|---:|---:|']
for h,r in results.items():
 d=r['magnitude'];lines.append(f"| {h}s | {d['mae']:.5f} | {d['rmse']:.5f} | {d['r2']:.5f} | {d['bias_pred_minus_actual']:.5f} | {d['std_ratio']:.4f} | {d['prediction_slope']:.4f} | {r['training_median_baseline']['test_mae']:.5f} |")
lines+=['','| Horizon | Magnitude range | N | MAE | Bias |','|---|---|---:|---:|---:|']
for h,r in results.items():
 for b in r['magnitude_bins']:lines.append(f"| {h}s | {b['range']} | {b['n']} | {b['mae']} | {b['bias']} |")
lines+=['','Strong-event diagnostic is prespecified as catalog magnitude ≥5.0 for this experiment; it is not an operational severity threshold.','']
for h,r in results.items():lines.append(f"- {h}s: {json.dumps(r['strong_magnitude'])}")
lines+=['','## Timing','', f"Hardware: {checkpoints[1]['environment']}. Batch-one latency uses 25 warmups and 200 timed GPU-synchronized forward passes, preloaded input; includes fixed transform, excludes disk/transfer/acquisition/trigger. Latency is subject to device load and is not a streaming benchmark.",'', '| Horizon | Median / p95 inference ms | S minus cutoff min / p05 / median / max seconds |','|---|---|---|']
for h,r in results.items():
 t=r['timing'];l=t['inference_ms'];a=t['same_station_s_minus_cutoff_seconds'];lines.append(f"| {h}s | {l['median']:.4f} / {l['p95']:.4f} | {a['min']:.4f} / {a['p05']:.4f} / {a['median']:.4f} / {a['max']:.4f} |")
lines+=['','S-minus-cutoff = saved continuous S time − (saved oracle P time + horizon), for eligible earthquakes only. It is a same-station synthetic interval, not operational warning lead time. The windows include 1 second of pre-P context; no streaming trigger delay was measured.','', '## Generalization and shortcut diagnostics','', '| Horizon | Train / validation / test F1 | Train / validation / test magnitude MAE | Post-P-zeroed F1 / ROC-AUC |','|---|---|---|---|']
for h,r in results.items():
 g=r['generalization'];a=r['post_P_zeroed_diagnostic'];lines.append(f"| {h}s | {g['train_f1']:.4f} / {g['validation_f1']:.4f} / {r['detection']['f1']:.4f} | {g['train_mag_mae']:.4f} / {g['validation_mag_mae']:.4f} / {r['magnitude']['mae']:.4f} | {a['f1']:.4f} / {a['roc_auc']:.4f} |")
lines+=['', 'Post-P-zeroing is a distribution-shift diagnostic of a frozen model, not proof of leakage or a pure pre-P-trained baseline. High retained discrimination would suggest noise/background or alignment shortcuts. Low retained discrimination does not exclude synthetic source/amplitude shortcuts. No direct target metadata is used; prior causality approval is reused, not rerun.','']
for h,r in results.items():
 lines.append(f"- {h}s noise-profile false-positive rates: {json.dumps(r['noise_profile_errors'])}")
 d=r['magnitude'];strong=r['strong_magnitude'];lines.append(f"- {h}s magnitude spread ratio {d['std_ratio']:.3f}, slope {d['prediction_slope']:.3f}; actual range {d['actual_range']}, predicted range {d['predicted_range']}. Values below one indicate prediction compression. Strong-event signed bias: {strong['bias']}.")
lines+=['', 'Results establish performance only on this generator and split. Event-disjoint synthetic testing cannot establish generalization to real instruments, noise, source physics or causal picked windows. Oracle alignment itself is privileged. Synthetic amplitude/distance assumptions can make magnitude easier or ambiguous. Tail estimates with small counts are uncertain; no confidence intervals or multiple seeds were evaluated. Test outputs were inspected only after all three checkpoint selections and did not change architecture, thresholds or settings.','', '## Essential plots (PNG, 300 DPI)','']
for name in ['training_validation_loss.png','actual_predicted_magnitude.png','confusion_matrices.png','magnitude_residuals.png','magnitude_mae_comparison.png','detection_f1_comparison.png']:lines.append(f'- [plots/{name}](plots/{name})')
lines+=['','## Preservation and reproduction','', 'Only the two notebooks, three best checkpoints, six plots and this report were created under cnn_eew_training/. No duplicate dataset or standalone Python script was written. Checkpoints contain model weights, configuration, selected epoch, complete small training history, train-only target transforms, input hashes and environment; no extra last/epoch checkpoints. Re-execution refuses to overwrite saved models/plots/report. Evaluation imports only the common-definition cell, never the training loop.','',f"Before/after hashes match for all {len(protection)} opened original dependencies (three HDF5 files and readiness report). Other existing files were not opened for writing. The checkpoint records those hashes. Selected validation losses reproduced after checkpoint reload.",'', '**STOP: standalone CNN training and evaluation complete.**']
with (OUT/'final_cnn_eew_report.md').open('x',encoding='utf-8') as f:f.write('\n'.join(lines)+'\n')
print('Six 300-DPI plots and final report saved. Four protected dependencies unchanged.')


Six 300-DPI plots and final report saved. Four protected dependencies unchanged.


In [4]:
# Interpret saved results without refitting, inference or additional test selection.
from pathlib import Path
report=(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'dataset_2.csv').is_file()) / 'cnn/cnn wave/eew_causal_v3/cnn_eew_training/final_cnn_eew_report.md')
text=report.read_text(encoding="utf-8")
interpretation="""Observed limitations: all three regressors show substantial prediction compression (predicted/actual SD 0.424–0.551; slopes 0.167–0.282), with many predictions near magnitude 3.5–3.8. They underpredict 94.7–97.7% of the 131 eligible magnitude≥5 events, with mean biases −1.16 to −1.38 magnitude units. The small training/validation/test MAE gaps do not establish external generalization: test R² remains only 0.146–0.290, and gains over a training-median predictor are modest.

Post-P-zeroed ROC-AUC remains 0.592–0.620, and false-positive rates differ substantially by noise family (colored pink is much easier than the other families). Together these are evidence consistent with background-family shortcuts; the distribution-shift probe does not isolate causation or prove future leakage. Direct target-feature leakage was not observed in the loader/model interface. No settings were changed in response to these results.

"""
anchor="Results establish performance only on this generator and split."
assert anchor in text
report.write_text(text.replace(anchor,interpretation+anchor,1),encoding="utf-8")
print("Recorded magnitude compression, strong-event underprediction and possible background shortcuts. No retraining.")


Recorded magnitude compression, strong-event underprediction and possible background shortcuts. No retraining.
